# 1장 · 놀람의 크기 — 정보를 숫자로 재다 😲

웹 챕터 **"놀람의 크기"** 와 짝을 이루는 Python 실습입니다.
웹에서 게이지로 잡은 감(놀람 I = log₂(1/p))을, 여기서는 **직접 계산·그래프·검증**으로 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch1')

## 1. 놀람 계산기: `np.log2` 한 줄이면 끝

놀람 공식은 $I = \log_2 \frac{1}{p}$. NumPy 의 `np.log2` 가 "2를 몇 번 곱하나"를 대신 세 줍니다.
일상의 사건 몇 개를 재 봅시다.

In [ ]:
import numpy as np

def surprise(p):
    """확률 p 인 사건의 놀람 (비트). I = log2(1/p)"""
    return np.log2(1 / p)

events = {
    '☀️ 내일 해가 뜬다      ': 1.0,
    '🪙 동전 앞면           ': 1 / 2,
    '🎲 주사위 6            ': 1 / 6,
    '🃏 카드 한 장이 ♠A     ': 1 / 52,
    '🎂 생일을 한 번에 맞힘 ': 1 / 365,
    '🎰 로또 1등            ': 1 / 8145060,
}

for name, p in events.items():
    print(f'{name} p = {p:>12.10f} → 놀람 {surprise(p):6.2f} 비트')

확실한 일(p=1)은 **0비트**, 로또 1등은 **약 23비트**로, 반반 질문 23번 어치의 놀람입니다.
(23비트가 커 보이지 않나요? 후보 2²³ ≈ 800만 개를 하나로 좁히는 정보라고 읽으면 됩니다.)

## 2. 놀람 곡선: p → 1 이면 죽고, p → 0 이면 폭발

웹의 놀람 게이지에서 본 곡선을 직접 그려 봅니다.

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

p = np.linspace(0.005, 1, 500)
I = np.log2(1 / p)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p, I, color='#6366f1', lw=2.5)

# 눈에 익은 점 몇 개 찍기
for prob, name in [(1.0, 'p=1 → 0비트'), (0.5, '동전 1비트'), (1/6, '주사위 2.58비트')]:
    ax.plot(prob, np.log2(1/prob), 'o', color='#4f46e5', ms=8)
    ax.annotate(name, (prob, np.log2(1/prob)), textcoords='offset points', xytext=(8, 6), fontsize=10)

ax.set_xlabel('확률 p')
ax.set_ylabel('놀람 I (비트)')
ax.set_title('놀람 곡선 I = log₂(1/p): 희귀할수록 절벽처럼 치솟는다')
ax.grid(True, color='#eef2f7')
plt.show()

## 3. "확률은 곱, 놀람은 합": 코드로 검증

log 를 쓰는 진짜 이유를 무작위 실험 1000번으로 확인합니다.
독립인 두 사건의 확률을 아무렇게나 뽑아도, **결합 사건의 놀람 = 두 놀람의 합**이 항상 성립할까요?

In [ ]:
rng = np.random.default_rng(42)

p1 = rng.uniform(0.01, 1, size=1000)   # 사건 A 의 확률 1000개
p2 = rng.uniform(0.01, 1, size=1000)   # 사건 B 의 확률 1000개

together = surprise(p1 * p2)           # 함께 일어난 사건의 놀람 (확률은 곱)
added = surprise(p1) + surprise(p2)    # 각 놀람의 합

print('최대 오차:', np.abs(together - added).max())
print('1000쌍 모두 일치?', np.allclose(together, added))

# 예시 하나만 눈으로: 동전 앞면(1/2) + 주사위 6(1/6)
print(f'\n동전 1비트 + 주사위 {surprise(1/6):.4f}비트 = {1 + surprise(1/6):.4f}비트')
print(f'함께 일어날 확률 1/12 의 놀람     = {surprise(1/12):.4f}비트')

## 4. AI 미리보기: 모델의 자신감과 손실

LLM 학습에서 정답 토큰의 손실은 **모델이 정답에 줬던 확률 q 의 놀람** $-\log q$ 입니다.
모델의 자신감(q)에 따라 손실이 얼마나 달라지는지 그려 봅시다.

In [ ]:
q = np.array([0.9, 0.5, 0.1, 0.01, 0.001])   # 모델이 정답 토큰에 준 확률
loss = np.log2(1 / q)                          # 손실 = 놀람 (비트)

labels = [f'q={v}' for v in q]
colors = ['#10b981', '#6366f1', '#f59e0b', '#f97316', '#ef4444']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, loss, color=colors)
ax.bar_label(bars, fmt='%.2f')
ax.set_xlabel('모델이 정답 토큰에 준 확률 q')
ax.set_ylabel('손실 = 놀람 (비트)')
ax.set_title('자신 있게 맞히면 거의 0, "설마" 했으면 폭발: 희귀 토큰이 크게 혼나는 이유')
ax.grid(True, axis='y', color='#eef2f7')
plt.show()

## 5. 직접 풀어보기 ✏️

동전을 **10번 던져 전부 앞면**이 나올 확률은 $(1/2)^{10} = 1/1024$.
이 사건의 놀람은 몇 비트일까요? (웹 챕터의 어려움 문제와 같은 계산)

In [ ]:
# TODO: 아래 answer 를 여러분의 답으로 바꾸고 실행하세요.
# 힌트: surprise((1/2)**10) — 또는 "1비트짜리 놀람이 10번 더해진다"로 암산!

answer = surprise((1 / 2) ** 10)
print('내 답:', answer)
assert abs(answer - 10) < 1e-9, '다시 생각해 보세요 — 앞면 하나(1비트)가 10번 독립으로 쌓이면?'
print('✅ 잘했어요! 확률의 곱셈(1/1024)이 놀람의 덧셈(10비트)이 됐습니다.')

---
**오늘의 한 줄**: 놀람 I = log₂(1/p). 확실하면 0, 희귀하면 폭발, 독립이면 더해진다.

다음 챕터에서는 이 눈금으로 **압축**을 이해합니다. 자주 나오는 글자에 짧은 코드를 주면
왜 이득인지, 그리고 GPT 의 토큰화(BPE)가 왜 압축인지. 🙂